# Phase 9: The API and streaming

FastAPI puts the workflow behind HTTP endpoints, so any app can use it. Progress is streamed as server-sent events: one message per finished step.

**Before you run the cells:** open a new terminal in VS Code, run the command below, and leave it running. Stop it later with Ctrl+C.

```bash
uv run uvicorn community_voice.api.main:app --port 8800
```

You can also try the endpoints in your browser at http://127.0.0.1:8800/docs.

In [ ]:
import json

import httpx

BASE = "http://127.0.0.1:8800"


def stream(path, payload):
    """POST and print each event as it arrives. Returns all events."""
    events = []
    name = None
    with httpx.stream("POST", BASE + path, json=payload, timeout=180) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line.startswith("event:"):
                name = line.split(":", 1)[1].strip()
            elif line.startswith("data:"):
                data = json.loads(line.split(":", 1)[1])
                events.append((name, data))
                shown = data.get("message") or data.get("report_id") or data
                print(f"{name:8}", str(shown)[:100])
    return events

## 1. Is it running?

In [ ]:
print(httpx.get(BASE + "/health").json())

## 2. Submit a report and watch it stream

The critical Tembisa report skips the web check, so the path is predictable. The stream ends when the report is waiting for a reviewer.

In [ ]:
report = {
    "text": "After last night's storm there are live wires hanging low over "
    "the road near Tembisa Hospital. Children walk under them to get to school."
}
events = stream("/reports", report)
report_id = events[0][1]["report_id"]

## 3. The review queue and the report

In [ ]:
for item in httpx.get(BASE + "/reviews").json():
    print(item["id"], item["urgency"], item["status"])
detail = httpx.get(f"{BASE}/reports/{report_id}").json()
print("Waiting for:", detail["waiting_for"])

## 4. Send the review

The reviewer approves. Watch the Drafter work, then the stream pauses again for the draft.

In [ ]:
events = stream(f"/reports/{report_id}/review", {"action": "approve"})
draft = events[-1][1]["payload"]["draft"]["body"]
print(draft)

## 5. Approve the draft

This call returns plain JSON. Read the draft first.

In [ ]:
response = httpx.post(
    f"{BASE}/reports/{report_id}/draft", json={"action": "approve"}, timeout=60
)
print(response.json()["status"])

## 6. Answering at the wrong time

The report is finished, so another draft decision is refused.

In [ ]:
response = httpx.post(
    f"{BASE}/reports/{report_id}/draft", json={"action": "approve"}, timeout=60
)
print(response.status_code, response.json())

## What to notice

- Each step arrived as soon as it finished, not all at the end.
- The API only moves a report forward when it is waiting for that decision.
- A screen can use the same calls: stream the steps, show the queue, send decisions.